# 🐘 hadoop02_fault_tolerance.ipynb
### Phase 7 — Hadoop + HDFS · Part 2
**Topics:** Fault Tolerance · Write/Read Flow · Secondary NameNode · Standby NameNode · HA Architecture

---
```
👁️  WATCH    →  Understand the concept
⌨️  BUILD    →  Structure it yourself
🗣️  EXPLAIN  →  Write it in plain English
```
---
## 🗣️ EXPLAIN BLOCK — Fill this LAST

**Q1: What is Hadoop in one sentence and what problem does it solve?**
`YOUR ANSWER:`File storage and operation of big data 

**Q2: What is HDFS and how is it different from a normal file system?**
`YOUR ANSWER:`it can made of clusters 

**Q3: What is the role of NameNode vs DataNode? What happens if NameNode goes down?**
`YOUR ANSWER:`

**Q4: What is replication factor and why is 3 the default?**
`YOUR ANSWER:`

**Q5: What is rack awareness and why does it matter for fault tolerance?**
`YOUR ANSWER:`

---
---
# SECTION 1 — Carry Forward Fixes from hadoop01
---
> Fix these 3 things from last notebook before moving forward.

## 🔧 FIX 1 — Block Size vs File Size Limit

**Q: Does a normal file system have a 128KB file size limit? What is 128MB in HDFS context?**
`YOUR CORRECTED ANSWER:`
128 Mb is the default block size in HDFS

**Q: Does a 50KB file take up a full 128MB block on disk in HDFS?**
`YOUR CORRECTED ANSWER:`
no it only takes 50 kb in 128 mb block

## 🔧 FIX 2 — Replication Placement (Correct Version)

**With replication=3 and rack awareness, where exactly are the 3 copies placed?**
```
Copy 1 → Rack 1, Node 1  reason: 2 dn in same rack
Copy 2 → Rack 1, Node 2   reason: 2 dn in same rack
Copy 3 → Rack 2, Node 3   reason: 1 dn in other rack


## 🔧 FIX 3 — NameNode Single Point of Failure

**Q: Is NameNode a single point of failure? What does that mean exactly?**
`YOUR ANSWER:`
if name node fails we will lost the meta data 
**Q: What were the 2 solutions Hadoop introduced to solve this?**
```
Solution 1: secondary NN → what it does: keep fsi image and edit logs
Solution 2: stand by name node → what it does: replica of primary data node
```

---
---
# SECTION 2 — HDFS Write and Read Flow
---

## ⌨️ BUILD 10 — Data Write Flow

> Fill every blank. You should know this from the videos.

```
Step 1: Client contacts _____________ 
        and says: I want to write file X

Step 2: NameNode checks _____________
        and responds with: _____________
        (what exactly does it send back?)

Step 3: Client splits file into _____________

Step 4: Client writes Block 1 to _____________
        (which DataNode? how is it chosen?)

Step 5: DataNode 1 replicates to _____________
        (this is called pipeline replication)

Step 6: DataNode 2 replicates to _____________

Step 7: Acknowledgement travels back:
        DN3 → DN2 → DN1 → _____________

Step 8: NameNode updates _____________
        to record new block locations
```

**Key question: Why does the client write directly to DataNode and NOT through NameNode?**
`YOUR ANSWER:`

## ⌨️ BUILD 11 — Data Read Flow

```
Step 1: Client contacts _____________
        and says: I want to read file X

Step 2: NameNode returns _____________
        (what exactly? block list? DataNode addresses?)

Step 3: Client contacts _____________ directly
        Why NOT through NameNode? _____________

Step 4: Client reads Block 1 from _____________ DataNode
        How does it decide which copy to read? _____________

Step 5: If that DataNode is slow or down:
        Client automatically reads from _____________

Step 6: Client assembles all blocks into _____________
```

**Key insight: What would happen if ALL reads/writes went through NameNode?**
`YOUR ANSWER:`

## 🧠 THINK — Write vs Read

**Q: In the write flow, DataNodes acknowledge in pipeline: DN3→DN2→DN1→Client. Why this direction and not DN1→DN2→DN3→Client?**
`YOUR ANSWER:`

**Q: A client is reading a 1GB file. Block 4 is on Node3 which is suddenly very slow. What happens?**
`YOUR ANSWER:`

**Q: Two clients try to write the same file at the same time. What does HDFS do?**
`YOUR ANSWER:`

---
---
# SECTION 3 — Fault Tolerance
---

## ⌨️ BUILD 12 — Fault Tolerance Scenarios

**Q: With rack awareness + replication=3, how many simultaneous node failures can HDFS survive?**
`YOUR ANSWER:`
2 node failures
**Q: Entire Rack 1 loses power. Replication=3, rack awareness=ON. Is any data lost? Why?**
`YOUR ANSWER:`
no as due to rack awarness and replication factor 3 it will get  a copy will be stored in diff reck

**Q: Same scenario but rack awareness=OFF — all 3 copies on same rack. What happens?**
`YOUR ANSWER:`
all data get wiped as every single node is on same rack

**Q: NameNode crashes at 11:47 AM. Last checkpoint was 10:00 AM. What data is lost?**
`YOUR ANSWER:`
10-11.47 am whatever edit logs are there and may be some recent files editlogs will be added to fsi image in seondary name node

---
---
# SECTION 4 — Secondary NameNode
---
> You are currently watching this lecture. Fill this section after watching.

## ⌨️ BUILD 13 — FSImage and Edit Logs

**Q: What is FSImage? Where is it stored?**
`YOUR ANSWER:`last save point of data

**Q: What is Edit Log? What does it record?**
`YOUR ANSWER:`
whatever changes done after the fsi image created 
**Q: When NameNode restarts — what does it do with FSImage and Edit Log?**
```
Step 1: fsi image + editlogs that are currently happening
Step 2: next day (last day fsi image + prev day edit logs)
Step 3: _______________
```

**Q: What happens if Edit Log is never merged? Why is that a problem?**
`YOUR ANSWER:`
any changes after fsi image created will not get saved

## ⌨️ BUILD 14 — Secondary NameNode Role

**Q: What is the ONLY job of Secondary NameNode?**
`YOUR ANSWER:`maintaing fsi image and editlogs record

**Q: Is Secondary NameNode a backup for NameNode? YES or NO and why?**
`YOUR ANSWER:`no because if it fails secondary name node doesnt replace it it just keep the track

**Q: Fill the checkpoint process step by step:**
```
Step 1: Secondary NN tells NameNode: what has been changes with edit logs

Step 2: Secondary NN copies from NameNode:
        - FSI image
        - edit logs

Step 3: Secondary NN does: add fsi image and editlogs

Step 4: Result sent back to NameNode: nect day morning
Step 5: Edit Log on NameNode: _______________
```

**Q: How often does this checkpoint happen by default?**
`YOUR ANSWER:`
10.5 minutes

## ⌨️ BUILD 15 — Game Save Analogy (Your Own Words)

> You already understood this perfectly in chat. Write it here so it stays in this notebook forever.

```
FSImage     = last save game
Edit Log    = changes made after the save
Secondary   =  saves games+changes made after the save
NameNode
Crash       = restart from secondar name node
Recovery    = 
Data loss   = _______________
```

## 🧠 THINK — Secondary NameNode Limitations

**Q: NameNode crashes at 2:47 PM. Last checkpoint was 2:00 PM. HDFS is down until 3:10 PM. What was lost? What is the downtime?**
`YOUR ANSWER:`2.47 to 3.10 data was lost 23 min is down time

**Q: For a bank that processes transactions 24/7, is Secondary NameNode enough? Why not?**
`YOUR ANSWER:`no because all trancation need to be update we cant afford the downtime so we use standby name node there

**Q: What is the fundamental limitation of Secondary NameNode that Standby NameNode solves?**
`YOUR ANSWER:`it cant sore after gets down where are standby is like backup the time primary name node fails secondary name node comes into action

---
---
# SECTION 5 — Standby NameNode and HA
---
> Fill after watching Standby NameNode and HA Architecture lectures.

## ⌨️ BUILD 16 — Secondary vs Standby NameNode

| Factor | Secondary NameNode | Standby NameNode |
|--------|-------------------|------------------|
| Purpose |saves FSI Image and edit logs| backup acts as a name node only |
| Always running? | no|yes |
| Has live metadata? |no| yes|
| Failover time | n/a|i guess 5-10 min |
| Data loss on failover |after primary name node crashes|nothing |
| Used in production today? |no|yes |

## ⌨️ BUILD 17 — HA Architecture

**Q: What are the 3 main components of Hadoop HA mode?**
```
1. Journal Node → role: keep sync between active NN and standby NN
2. zookeeper Failover   → role: keep sending signals to zookeper if any of the NN fails
3. Zookeeper → role: monitors everything is working fine or not
```

**Q: What is JournalNode and why does HA need it?**
`YOUR ANSWER:`it keeps bot Name node and standby name node in sync 

**Q: What is Zookeeper's role in HA failover?**
`YOUR ANSWER:`it sends signal to leader

**Q: What is split-brain in HA and why is it dangerous?**
`YOUR ANSWER:`split brain thinks one node fails but its working which can led to meta data corruption,file system duplicay ,block failure

## ⌨️ BUILD 18 — HA Failover Step by Step

```
Normal state:
Active NameNode   → handling all client requests
Standby NameNode  → replicates all thing by journal node
Zookeeper         → gets signal from both name nodes

Active NameNode crashes:
Step 1: Zookeeper detects: active node gets failed
Step 2: Zookeeper triggers: zookeper failover 
Step 3: Standby becomes: standby node become active
Step 4: Clients reconnect to: stand by name node
Step 5: Total downtime: zero
```

## 🧠 THINK — HA Design

**Q: Why does HA need an ODD number of JournalNodes (3, 5, 7)?**
`YOUR ANSWER:`

**Q: Can you have 2 Active NameNodes at the same time? What would happen?**
`YOUR ANSWER:`

**Q: A company is choosing between Secondary NameNode setup and HA setup. What questions would you ask to help them decide?**
`YOUR ANSWER:`

---
---
# 🏋️ PRACTICE TASKS
---

## Practice Task 1 — Storage Planning

> Company requirements:
> - Raw log files: 500GB per day
> - Keep 90 days of logs
> - Replication factor: 3, Block size: 128MB

**Q1: Total raw data after 90 days?**
`YOUR ANSWER:`

**Q2: Total HDFS storage needed with replication?**
`YOUR ANSWER:`

**Q3: Total number of 128MB blocks?**
`YOUR ANSWER:`

**Q4: Minimum DataNodes if each has 10TB usable disk?**
`YOUR ANSWER:`

**Q5: Boss wants 40% cost reduction. What are your 3 options?**
`YOUR ANSWER:`

In [0]:
# Practice Task 2 — HDFS Storage Simulator
# Build a function that:
# 1. Takes filename and file_size_mb
# 2. Splits into 128MB blocks
# 3. Assigns copies with rack awareness:
#    Copy 1 → random node in Rack 1
#    Copy 2 → different node in Rack 1
#    Copy 3 → random node in Rack 2
# 4. Prints full storage map
# 5. Simulate node failure — show under-replicated blocks

import random
import math

cluster = {
    'Rack1': ['Node1', 'Node2', 'Node3'],
    'Rack2': ['Node4', 'Node5', 'Node6']
}

def simulate_hdfs_write(filename, file_size_mb, block_size_mb=128, replication=3):
    # Step 1: Calculate number of blocks
    # Step 2: For each block assign 3 copies with rack awareness
    # Step 3: Print storage map
    # Step 4: Return storage_map dict
    # YOUR CODE:
    pass

def simulate_node_failure(storage_map, failed_node):
    # Find all blocks that had a copy on failed_node
    # Print which blocks are now under-replicated
    # YOUR CODE:
    pass

storage = simulate_hdfs_write('sales_data.csv', 300)
simulate_node_failure(storage, 'Node2')

## Practice Task 3 — Interview Questions (No Notes)

**Q1: File=1GB, Block=128MB, Replication=3. How many total block copies in HDFS?**
`YOUR ANSWER:`

**Q2: NameNode goes down. Can existing data still be read? Why?**
`YOUR ANSWER:`

**Q3: 5 DataNodes. Block A: copy1=Node1, copy2=Node2, copy3=Node3. Node1 AND Node2 crash. Is Block A lost?**
`YOUR ANSWER:`

**Q4: Why is HDFS bad for storing millions of tiny 1KB files?**
`YOUR ANSWER:`

**Q5: HDFS is write-once, read-many. What does this mean and why?**
`YOUR ANSWER:`

**Q6: What is the difference between Secondary NameNode and Standby NameNode?**
`YOUR ANSWER:`

**Q7: NameNode crashes at 3PM. Last checkpoint at 2PM. What happens? What is lost?**
`YOUR ANSWER:`

---
---
# ✅ COMPLETION CHECKLIST

| Task | Done? |
|------|-------|
| 🔧 FIX 1 — Block size vs file size limit corrected | |
| 🔧 FIX 2 — Replication placement corrected | |
| 🔧 FIX 3 — Single point of failure answered | |
| ⌨️ BUILD 10 — Write flow all 8 steps filled | |
| ⌨️ BUILD 11 — Read flow all 6 steps filled | |
| ⌨️ BUILD 12 — Fault tolerance scenarios | |
| 👁️ Watched: DataNode Failure Temporary | |
| 👁️ Watched: DataNode Failure Permanent | |
| 👁️ Watched: Secondary NameNode | |
| ⌨️ BUILD 13 — FSImage and Edit Logs | |
| ⌨️ BUILD 14 — Secondary NameNode checkpoint process | |
| ⌨️ BUILD 15 — Game save analogy in your words | |
| 👁️ Watched: Standby NameNode | |
| 👁️ Watched: Hadoop HA Architecture | |
| ⌨️ BUILD 16 — Secondary vs Standby comparison table | |
| ⌨️ BUILD 17 — HA Architecture components | |
| ⌨️ BUILD 18 — HA failover step by step | |
| 🏋️ Practice Task 1 — Storage planning all 5 Qs | |
| 🏋️ Practice Task 2 — HDFS simulator coded and running | |
| 🏋️ Practice Task 3 — All 7 interview questions | |
| 🗣️ All 5 EXPLAIN questions answered at top | |
| 🧠 All THINK questions answered | |

---
**Checklist full → send back → get hadoop03_cluster_commands.ipynb 🚀**